# LCA analizė – pilnas pavyzdys

Šis notebook'as demonstruoja:
1. Duomenų įkėlimą iš CSV ir EPD International API
2. Poveikio kategorijų palyginimą
3. GWP vizualizaciją (stulpelinė, radaro, medžio diagramos)
4. Produktų normalizavimą ir reitingavimą

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.express as px
import plotly.graph_objects as go

from lca_analysis.parsers import parse_excel, EPDApiClient, SearchFilters
from lca_analysis.processing import products_to_dataframe, wide_format
from lca_analysis.visualization import bar_comparison, radar_chart, treemap_gwp

DATA = Path('../data')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
print('Bibliotekos įkeltos ✓')

## 1. Duomenų įkėlimas

In [ ]:
# --- 1a. CSV failas (pavyzdiniai duomenys) ---
csv_path = DATA / 'raw/manual/example_products.csv'
products_local = parse_excel(csv_path)
print(f'Įkelti {len(products_local)} produktai iš CSV')
for p in products_local:
    gwp = p.gwp()
    print(f'  {p.id}: {p.name[:40]:<40} GWP={gwp:>8.1f} kg CO₂ eq / {p.declared_unit}')

In [ ]:
# --- 1b. EPD International API (reikalingas internetas) ---
# Naudokite šį bloką tikroms EPD iš api.environdec.com

try:
    client = EPDApiClient()
    filters = SearchFilters(query="concrete", page_size=10)
    raw_page = client.search(filters)
    total = raw_page.get('totalCount', 0)
    print(f'EPD API: rasta {total} įrašų pagal "concrete"')

    # Paversti į Product objektus
    products_api = [client.to_product(r) for r in raw_page.get('data', [])]
    print(f'Konvertuoti {len(products_api)} produktai')
    for p in products_api[:5]:
        print(f'  {p.id[:20]:<20} {p.name[:40]:<40} GWP={p.gwp()}')
except Exception as e:
    print(f'API nepasiekiama: {e}')
    products_api = []

In [ ]:
# --- Sujungti visus šaltinius ---
all_products = products_local + products_api
df = products_to_dataframe(all_products)
print(f'Iš viso: {df["id"].nunique()} unikalūs produktai, {len(df)} poveikio įrašai')
df.head()

## 2. GWP palyginimas (stulpelinė diagrama)

In [ ]:
gwp_df = (
    df[df['category'] == 'GWP']
    .groupby(['name', 'declared_unit'], as_index=False)['value']
    .first()
    .sort_values('value')
)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in gwp_df['value']]
bars = ax.barh(gwp_df['name'], gwp_df['value'], color=colors, edgecolor='white', linewidth=0.5)

# Reikšmių žymos
for bar, val in zip(bars, gwp_df['value']):
    ax.text(
        bar.get_width() + (5 if val >= 0 else -5),
        bar.get_y() + bar.get_height() / 2,
        f'{val:.1f}',
        va='center', ha='left' if val >= 0 else 'right', fontsize=8
    )

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('GWP [kg CO₂ eq / funk. vienetui]', fontsize=11)
ax.set_title('Šiltnamio efektą sukeliančių dujų poveikis (A1–A3)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(DATA / 'results/gwp_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Išsaugota: data/results/gwp_comparison.png')

## 3. Radaro diagrama – kelios poveikio kategorijos

In [ ]:
# Normalizuojame kiekvieną kategoriją pagal maksimumą (0–1 skalė)
wide = wide_format(df)
categories = ['GWP', 'AP', 'EP', 'POCP', 'ADPF']
available_cats = [c for c in categories if c in wide.columns]

# Normalizavimas
wide_norm = wide.copy()
for cat in available_cats:
    max_val = wide_norm[cat].abs().max()
    if max_val > 0:
        wide_norm[cat] = wide_norm[cat] / max_val

# Pasirinkti 4 produktus palyginimui
compare_ids = ['p001', 'p002', 'p005', 'p006']  # betonas, CLT, akmens vata, plytos
compare_ids = [i for i in compare_ids if i in wide_norm['id'].values]

fig = go.Figure()
colors_radar = px.colors.qualitative.Set2
for idx, pid in enumerate(compare_ids):
    row = wide_norm[wide_norm['id'] == pid].iloc[0]
    vals = [row.get(c, 0) or 0 for c in available_cats]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=available_cats + [available_cats[0]],
        name=row['name'][:30],
        fill='toself',
        line=dict(color=colors_radar[idx % len(colors_radar)]),
        fillcolor=colors_radar[idx % len(colors_radar)],
        opacity=0.4,
    ))

fig.update_layout(
    title='Poveikio kategorijų palyginimas (normalizuota)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    width=700, height=550,
)
fig.write_html(str(DATA / 'results/radar_comparison.html'))
fig.show()
print('Išsaugota: data/results/radar_comparison.html')

## 4. GWP medžio diagrama (Treemap)

In [ ]:
# Tik teigiamos GWP reikšmės (neigiamos – anglies kaupimas) – atskira diagrama
gwp_pos = df[(df['category'] == 'GWP') & (df['value'] > 0)].copy()

fig = px.treemap(
    gwp_pos,
    path=['source', 'name'],
    values='value',
    color='value',
    color_continuous_scale='RdYlGn_r',
    title='GWP pasiskirstymas pagal šaltinį ir produktą (A1–A3, tik emisijos)',
    labels={'value': 'GWP [kg CO₂ eq]'},
)
fig.update_layout(width=900, height=550)
fig.write_html(str(DATA / 'results/treemap_gwp.html'))
fig.show()
print('Išsaugota: data/results/treemap_gwp.html')

## 5. Reitingavimas – mažiausias GWP per produktų grupę

In [ ]:
# Geriausių produktų pagal GWP lentelė
gwp_rank = (
    df[df['category'] == 'GWP']
    .assign(gwp_per_unit=lambda x: x['value'])
    [['name', 'declared_unit', 'gwp_per_unit', 'source']]
    .sort_values('gwp_per_unit')
    .reset_index(drop=True)
)
gwp_rank.index += 1
gwp_rank.columns = ['Produktas', 'Funk. vienetas', 'GWP [kg CO₂ eq]', 'Šaltinis']

# Eksportuoti į CSV
gwp_rank.to_csv(DATA / 'results/gwp_ranking.csv', index=True, encoding='utf-8-sig')
print('Išsaugota: data/results/gwp_ranking.csv')
gwp_rank.style \
    .background_gradient(subset=['GWP [kg CO₂ eq]'], cmap='RdYlGn_r') \
    .format({'GWP [kg CO₂ eq]': '{:.2f}'})

## 6. EPD International API – išplėstinė paieška

Šis blokas rodo, kaip ieškoti EPD pagal šalį ir kategoriją.

In [ ]:
# Ieškoti Lietuvos arba Baltijos šalių EPD
try:
    client = EPDApiClient()

    print('=== Lietuva (LT) ===')
    lt_filters = SearchFilters(country='LT', page_size=20)
    lt_page = client.search(lt_filters)
    print(f'Rasta: {lt_page.get("totalCount", 0)} EPD')
    for item in lt_page.get('data', [])[:5]:
        print(f'  [{item.get("id", "")[:15]}] {item.get("name", "")[:50]}')

    print('\n=== Betonas (visi) ===')
    concrete_filters = SearchFilters(query='concrete', page_size=5)
    concrete_products = client.fetch_products(concrete_filters, limit=5)
    for p in concrete_products:
        print(f'  {p.name[:50]:<50} GWP={p.gwp()}')

except Exception as e:
    print(f'API klaida: {e}')

## 7. Duomenų eksportas

In [ ]:
# Eksportuoti pilną duomenų rinkinį
output_path = DATA / 'results/all_products_long.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'Eksportuota: {output_path}')
print(f'Eilučių: {len(df)}, stulpelių: {len(df.columns)}')

# Platus formatas (wide) su visomis kategorijomis
wide_output = wide_format(df)
wide_output.to_csv(DATA / 'results/all_products_wide.csv', index=False, encoding='utf-8-sig')
print(f'Wide formatas: {DATA}/results/all_products_wide.csv')